# From Family to Population: The Bridge
## Interactive Simulation Showing How Population Patterns Emerge from Individual Inheritance

**The Key Insight:** Population genetics is just Mendelian genetics repeated thousands of times!

This notebook will help you SEE how the same mechanism works at different scales.

---

In [ ]:
# Install and import packages
!pip install ipywidgets matplotlib numpy pandas seaborn plotly -q

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from ipywidgets import interact, IntSlider, FloatSlider, Dropdown, widgets, interactive
from IPython.display import display, HTML, clear_output
import plotly.graph_objects as go
from plotly.subplots import make_subplots

sns.set_style("whitegrid")
np.random.seed(42)  # For reproducibility

print("✓ Ready to bridge from families to populations!")

---
## Part 1: ONE Family - Classic Mendelian Genetics

Let's start with what you know: A single cross

In [ ]:
def single_family_cross():
    """
    Simulate ONE Mendelian cross: Aa × Aa
    """
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    
    # LEFT: Punnett Square
    ax1.set_xlim(0, 4)
    ax1.set_ylim(0, 4)
    ax1.axis('off')
    ax1.set_title('Single Family: Aa × Aa', fontsize=16, fontweight='bold')
    
    # Draw Punnett square
    for i in range(3):
        ax1.plot([0.5, 3.5], [0.5+i, 0.5+i], 'k-', linewidth=2)
        ax1.plot([0.5+i, 0.5+i], [0.5, 3.5], 'k-', linewidth=2)
    
    # Labels
    ax1.text(0.25, 3.5, 'Mother\nAa', ha='center', va='center', fontsize=11, fontweight='bold')
    ax1.text(3.75, 3, 'Father Aa', ha='center', va='center', fontsize=11, fontweight='bold', rotation=270)
    
    ax1.text(1.5, 3.75, 'A (50%)', ha='center', fontsize=10, fontweight='bold')
    ax1.text(2.5, 3.75, 'a (50%)', ha='center', fontsize=10, fontweight='bold')
    ax1.text(0.25, 2.5, 'A (50%)', ha='center', fontsize=10, fontweight='bold')
    ax1.text(0.25, 1.5, 'a (50%)', ha='center', fontsize=10, fontweight='bold')
    
    # Genotypes
    genotypes = [['AA', 'Aa'], ['Aa', 'aa']]
    colors = [['#FF6B6B', '#FFD93D'], ['#FFD93D', '#6BCF7F']]
    
    for i in range(2):
        for j in range(2):
            ax1.add_patch(plt.Rectangle((1.5+j-0.45, 2.5-i-0.45), 0.9, 0.9, 
                                       facecolor=colors[i][j], alpha=0.6, edgecolor='black', linewidth=2))
            ax1.text(1.5+j, 2.5-i, genotypes[i][j], ha='center', va='center', 
                    fontsize=14, fontweight='bold')
    
    # Probabilities
    ax1.text(2, 0.2, '¼ AA : ½ Aa : ¼ aa', ha='center', fontsize=12, fontweight='bold',
            bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.5))
    
    # RIGHT: Expected outcome
    ax2.set_xlim(0, 5)
    ax2.set_ylim(0, 60)
    ax2.set_xlabel('Genotype', fontsize=12, fontweight='bold')
    ax2.set_ylabel('Percentage (%)', fontsize=12, fontweight='bold')
    ax2.set_title('Expected Offspring Distribution', fontsize=14, fontweight='bold')
    
    genotypes_list = ['AA', 'Aa', 'aa']
    percentages = [25, 50, 25]
    colors_bar = ['#FF6B6B', '#FFD93D', '#6BCF7F']
    
    bars = ax2.bar(genotypes_list, percentages, color=colors_bar, alpha=0.7, edgecolor='black', linewidth=2)
    ax2.set_ylim(0, 60)
    
    for bar, pct in zip(bars, percentages):
        height = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2., height + 2,
                f'{pct}%', ha='center', fontsize=12, fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    print("\n" + "="*70)
    print("INDIVIDUAL/FAMILY LEVEL THINKING:")
    print("="*70)
    print("Question: 'What genotype will THIS child have?'")
    print("Answer: '25% chance AA, 50% chance Aa, 25% chance aa'")
    print("\nThinking Mode: DETERMINISTIC PROBABILITY")
    print("  - We know the exact probabilities")
    print("  - Each child is an independent event")
    print("  - We're predicting one specific outcome")
    print("="*70)

single_family_cross()

---
## Part 2: THE ZOOM-OUT - From 1 Family to 10,000

**This is where the magic happens!** Watch how the same mechanism creates population patterns.

In [ ]:
def simulate_families(num_families, offspring_per_family=4):
    """
    Simulate multiple Aa × Aa families
    """
    # For each family, simulate offspring
    # Each offspring: 25% AA, 50% Aa, 25% aa
    total_offspring = num_families * offspring_per_family
    
    # Generate random offspring
    genotype_codes = np.random.choice([0, 1, 2], size=total_offspring, p=[0.25, 0.5, 0.25])
    # 0 = AA, 1 = Aa, 2 = aa
    
    count_AA = np.sum(genotype_codes == 0)
    count_Aa = np.sum(genotype_codes == 1)
    count_aa = np.sum(genotype_codes == 2)
    
    return count_AA, count_Aa, count_aa, total_offspring

def visualize_zoom_out(num_families):
    """
    Show how patterns emerge as we zoom from families to populations
    """
    offspring_per_family = 4
    
    # Simulate
    count_AA, count_Aa, count_aa, total = simulate_families(num_families, offspring_per_family)
    
    # Calculate percentages
    pct_AA = (count_AA / total) * 100
    pct_Aa = (count_Aa / total) * 100
    pct_aa = (count_aa / total) * 100
    
    # Calculate allele frequencies
    freq_A = (2 * count_AA + count_Aa) / (2 * total)
    freq_a = (2 * count_aa + count_Aa) / (2 * total)
    
    # Create figure
    fig = plt.figure(figsize=(16, 10))
    gs = fig.add_gridspec(3, 2, hspace=0.4, wspace=0.3)
    
    # 1. Main bar chart - Observed vs Expected
    ax1 = fig.add_subplot(gs[0:2, :])
    
    genotypes = ['AA', 'Aa', 'aa']
    x = np.arange(len(genotypes))
    width = 0.35
    
    expected = [25, 50, 25]
    observed = [pct_AA, pct_Aa, pct_aa]
    colors = ['#FF6B6B', '#FFD93D', '#6BCF7F']
    
    bars1 = ax1.bar(x - width/2, expected, width, label='Expected (Mendelian)', 
                    color=colors, alpha=0.5, edgecolor='black', linewidth=2)
    bars2 = ax1.bar(x + width/2, observed, width, label='Observed (Your Data)',
                    color=colors, alpha=0.9, edgecolor='black', linewidth=2)
    
    ax1.set_xlabel('Genotype', fontsize=13, fontweight='bold')
    ax1.set_ylabel('Percentage (%)', fontsize=13, fontweight='bold')
    ax1.set_title(f'From Family to Population: {num_families} Families ({total} Total Offspring)', 
                 fontsize=15, fontweight='bold')
    ax1.set_xticks(x)
    ax1.set_xticklabels(genotypes)
    ax1.legend(fontsize=11)
    ax1.set_ylim(0, 60)
    ax1.grid(True, alpha=0.3, axis='y')
    
    # Add value labels
    for bars in [bars1, bars2]:
        for bar in bars:
            height = bar.get_height()
            ax1.text(bar.get_x() + bar.get_width()/2., height + 1,
                    f'{height:.1f}%', ha='center', fontsize=10, fontweight='bold')
    
    # Calculate deviation from expected
    deviations = [abs(o - e) for o, e in zip(observed, expected)]
    avg_deviation = np.mean(deviations)
    
    # Add deviation indicator
    if avg_deviation < 2:
        deviation_text = "EXCELLENT match!"
        deviation_color = 'green'
    elif avg_deviation < 5:
        deviation_text = "GOOD match"
        deviation_color = 'orange'
    else:
        deviation_text = "More variation (normal with small sample)"
        deviation_color = 'red'
    
    ax1.text(0.98, 0.95, f'Average deviation: {avg_deviation:.2f}%\n{deviation_text}',
            transform=ax1.transAxes, ha='right', va='top', fontsize=11,
            bbox=dict(boxstyle='round', facecolor=deviation_color, alpha=0.3))
    
    # 2. Thinking mode indicator
    ax2 = fig.add_subplot(gs[2, 0])
    ax2.axis('off')
    
    if num_families <= 10:
        thinking_mode = "FAMILY LEVEL"
        mode_desc = """Still thinking about individual families.
Variation is large.
Each family's outcome matters.
Focus: Specific predictions"""
        mode_color = 'lightblue'
    elif num_families <= 100:
        thinking_mode = "TRANSITIONAL"
        mode_desc = """Starting to see patterns emerge.
Variation decreasing.
Moving toward statistics.
Focus: Trends appearing"""
        mode_color = 'lightyellow'
    else:
        thinking_mode = "POPULATION LEVEL"
        mode_desc = """Now thinking in frequencies!
Patterns are stable.
Individual families don't matter.
Focus: Statistical distributions"""
        mode_color = 'lightgreen'
    
    thinking_text = f"""THINKING MODE: {thinking_mode}

{mode_desc}

Total Individuals: {total:,}
Sample Size: {"Small" if total < 100 else "Medium" if total < 1000 else "Large"}
"""
    
    ax2.text(0.5, 0.5, thinking_text, ha='center', va='center', fontsize=10,
            family='monospace',
            bbox=dict(boxstyle='round', facecolor=mode_color, alpha=0.5))
    
    # 3. Population genetics view (allele frequencies)
    ax3 = fig.add_subplot(gs[2, 1])
    ax3.axis('off')
    
    # Hardy-Weinberg calculation
    p = freq_A
    q = freq_a
    
    HW_AA = p**2 * 100
    HW_Aa = 2*p*q * 100
    HW_aa = q**2 * 100
    
    pop_text = f"""POPULATION GENETICS VIEW:

Allele Frequencies:
  p (freq of A) = {freq_A:.4f}
  q (freq of a) = {freq_a:.4f}
  p + q = {freq_A + freq_a:.4f} ✓

Hardy-Weinberg Prediction:
  p² (AA) = {HW_AA:.2f}%
  2pq (Aa) = {HW_Aa:.2f}%
  q² (aa) = {HW_aa:.2f}%

Same as Mendel's 1:2:1!
(when p = q = 0.5)
"""
    
    if num_families >= 100:
        pop_text += "\n✓ You're thinking like a population geneticist!"
    
    ax3.text(0.5, 0.5, pop_text, ha='center', va='center', fontsize=9.5,
            family='monospace',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    plt.tight_layout()
    plt.show()
    
    # Print interpretation
    print("\n" + "="*70)
    print(f"WITH {num_families} FAMILIES:")
    print("="*70)
    
    if num_families == 1:
        print("You're looking at ONE family (4 kids).")
        print("The outcome might not match 1:2:1 exactly - that's normal!")
        print("This is like flipping a coin 4 times - you might not get exactly 2 heads.")
    elif num_families <= 10:
        print("You're looking at a few families.")
        print("You can still see individual variation.")
        print("The pattern is starting to emerge, but there's still randomness.")
    elif num_families <= 100:
        print("You're in the TRANSITION zone!")
        print("Individual families matter less, patterns matter more.")
        print("The 1:2:1 ratio is becoming clearer.")
    else:
        print("You're now thinking about a POPULATION!")
        print("Individual families don't matter - you see the overall pattern.")
        print("The ratio is very close to 1:2:1.")
        print("\n🎉 THIS IS POPULATION GENETICS!")
        print("   You're now using allele frequencies (p and q)")
        print("   You're thinking statistically, not individually")
        print("   But the MECHANISM is still Mendelian inheritance!")
    
    print("\nKey Insight: The SAME mechanism (Aa × Aa) is operating at ALL scales!")
    print("="*70)

# Interactive widget
print("\n🔬 THE ZOOM-OUT SIMULATOR")
print("Drag the slider to see how population patterns emerge from family genetics\n")

interact(visualize_zoom_out,
         num_families=IntSlider(min=1, max=10000, step=1, value=1,
                               description='Number of Families:',
                               style={'description_width': 'initial'},
                               readout_format='d'));

---
## Part 3: The Bridge - Punnett Square to Hardy-Weinberg

**The Revelation:** Hardy-Weinberg IS just a population-scale Punnett square!

In [ ]:
def show_punnett_to_hw(freq_A):
    """
    Show that Hardy-Weinberg is just Punnett square with frequencies
    """
    freq_a = 1 - freq_A
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))
    
    # LEFT: Family Punnett Square
    ax1.set_xlim(0, 5)
    ax1.set_ylim(0, 5)
    ax1.axis('off')
    ax1.set_title('FAMILY LEVEL: One Punnett Square', fontsize=14, fontweight='bold')
    
    # Draw grid
    for i in range(3):
        ax1.plot([1, 4], [1+i, 1+i], 'k-', linewidth=2)
        ax1.plot([1+i, 1+i], [1, 4], 'k-', linewidth=2)
    
    # Labels
    ax1.text(2.5, 4.5, f'Father (p={freq_A:.2f}, q={freq_a:.2f})', ha='center', fontsize=11, fontweight='bold')
    ax1.text(0.5, 2.5, f'Mother\n(p={freq_A:.2f}\nq={freq_a:.2f})', ha='center', va='center', 
            fontsize=11, fontweight='bold')
    
    ax1.text(2, 4.2, f'A ({freq_A:.2f})', ha='center', fontsize=10)
    ax1.text(3, 4.2, f'a ({freq_a:.2f})', ha='center', fontsize=10)
    ax1.text(0.7, 3, f'A ({freq_A:.2f})', ha='center', fontsize=10)
    ax1.text(0.7, 2, f'a ({freq_a:.2f})', ha='center', fontsize=10)
    
    # Cells with probabilities
    cells = [
        [(2, 3, f'AA\np²={freq_A**2:.3f}', '#FF6B6B'),
         (3, 3, f'Aa\npq={freq_A*freq_a:.3f}', '#FFD93D')],
        [(2, 2, f'Aa\npq={freq_A*freq_a:.3f}', '#FFD93D'),
         (3, 2, f'aa\nq²={freq_a**2:.3f}', '#6BCF7F')]
    ]
    
    for row in cells:
        for x, y, text, color in row:
            ax1.add_patch(plt.Rectangle((x-0.45, y-0.45), 0.9, 0.9,
                                       facecolor=color, alpha=0.6, edgecolor='black', linewidth=2))
            ax1.text(x, y, text, ha='center', va='center', fontsize=9, fontweight='bold')
    
    # Formula
    ax1.text(2.5, 0.5, 'ONE family outcome', ha='center', fontsize=11,
            bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.5))
    
    # RIGHT: Population Hardy-Weinberg
    ax2.set_xlim(0, 5)
    ax2.set_ylim(0, 5)
    ax2.axis('off')
    ax2.set_title('POPULATION LEVEL: Hardy-Weinberg\n(= Giant Punnett Square!)', 
                 fontsize=14, fontweight='bold')
    
    # Draw grid
    for i in range(3):
        ax2.plot([1, 4], [1+i, 1+i], 'k-', linewidth=3)
        ax2.plot([1+i, 1+i], [1, 4], 'k-', linewidth=3)
    
    # Labels
    ax2.text(2.5, 4.5, f'ALL Males (p={freq_A:.2f}, q={freq_a:.2f})', ha='center', 
            fontsize=11, fontweight='bold')
    ax2.text(0.3, 2.5, f'ALL\nFemales\n(p={freq_A:.2f}\nq={freq_a:.2f})', ha='center', va='center',
            fontsize=11, fontweight='bold')
    
    ax2.text(2, 4.2, f'A (p={freq_A:.2f})', ha='center', fontsize=10, fontweight='bold')
    ax2.text(3, 4.2, f'a (q={freq_a:.2f})', ha='center', fontsize=10, fontweight='bold')
    ax2.text(0.7, 3, f'A (p={freq_A:.2f})', ha='center', fontsize=10, fontweight='bold')
    ax2.text(0.7, 2, f'a (q={freq_a:.2f})', ha='center', fontsize=10, fontweight='bold')
    
    # Same cells, same probabilities!
    for row in cells:
        for x, y, text, color in row:
            ax2.add_patch(plt.Rectangle((x-0.45, y-0.45), 0.9, 0.9,
                                       facecolor=color, alpha=0.6, edgecolor='black', linewidth=3))
            ax2.text(x, y, text, ha='center', va='center', fontsize=9, fontweight='bold')
    
    # Formula
    hw_text = f"p² + 2pq + q² = 1\n{freq_A**2:.3f} + {2*freq_A*freq_a:.3f} + {freq_a**2:.3f} = 1.000"
    ax2.text(2.5, 0.5, hw_text, ha='center', fontsize=11, family='monospace',
            bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.5))
    
    plt.tight_layout()
    plt.show()
    
    # Calculate genotype frequencies
    freq_AA = freq_A ** 2
    freq_Aa = 2 * freq_A * freq_a
    freq_aa = freq_a ** 2
    
    print("\n" + "="*70)
    print("THE BIG REVEAL: THEY'RE THE SAME!")
    print("="*70)
    print("\nFamily Punnett Square          →    Population Hardy-Weinberg")
    print("─────────────────────────────       ────────────────────────────")
    print("One father × One mother            All males × All females")
    print("Probabilities for one child        Frequencies in population")
    print("p and q are gamete chances         p and q are allele frequencies")
    print("\n" + "="*70)
    print("\nExpected Genotype Frequencies:")
    print(f"  AA (p²):  {freq_AA:.4f} = {freq_AA*100:.2f}%")
    print(f"  Aa (2pq): {freq_Aa:.4f} = {freq_Aa*100:.2f}%")
    print(f"  aa (q²):  {freq_aa:.4f} = {freq_aa*100:.2f}%")
    print(f"  Total:    {freq_AA + freq_Aa + freq_aa:.4f} ✓")
    print("\n💡 KEY INSIGHT: Hardy-Weinberg is just Mendelian genetics")
    print("   applied to an entire population!")
    print("="*70)

# Interactive demonstration
print("\n🎯 PUNNETT SQUARE = HARDY-WEINBERG")
print("Adjust the A allele frequency to see they're the same equation!\n")

interact(show_punnett_to_hw,
         freq_A=FloatSlider(min=0.1, max=0.9, step=0.05, value=0.5,
                           description='Frequency of A allele (p):',
                           style={'description_width': 'initial'}));

---
## Part 4: Multi-Generation Simulation

Now let's see how populations change over time (or don't change!)

In [ ]:
def simulate_generations(pop_size, generations, selection_aa=1.0):
    """
    Simulate multiple generations
    selection_aa: fitness of aa genotype (1.0 = no selection)
    """
    # Start with p = q = 0.5
    freq_A_history = [0.5]
    freq_a_history = [0.5]
    
    current_freq_A = 0.5
    
    for gen in range(generations):
        current_freq_a = 1 - current_freq_A
        
        # Generate population with HW frequencies
        freq_AA = current_freq_A ** 2
        freq_Aa = 2 * current_freq_A * current_freq_a
        freq_aa = current_freq_a ** 2
        
        # Apply selection (aa individuals have reduced fitness)
        if selection_aa < 1.0:
            # Relative fitness
            w_AA = 1.0
            w_Aa = 1.0
            w_aa = selection_aa
            
            # Mean fitness
            w_bar = freq_AA * w_AA + freq_Aa * w_Aa + freq_aa * w_aa
            
            # New frequencies after selection
            freq_AA = (freq_AA * w_AA) / w_bar
            freq_Aa = (freq_Aa * w_Aa) / w_bar
            freq_aa = (freq_aa * w_aa) / w_bar
        
        # Calculate new allele frequency
        current_freq_A = freq_AA + 0.5 * freq_Aa
        current_freq_a = freq_aa + 0.5 * freq_Aa
        
        freq_A_history.append(current_freq_A)
        freq_a_history.append(current_freq_a)
    
    return freq_A_history, freq_a_history

def visualize_generations(generations, selection_strength):
    """
    Visualize how allele frequencies change over generations
    """
    selection_aa = 1.0 - (selection_strength / 100)
    
    freq_A, freq_a = simulate_generations(1000, generations, selection_aa)
    
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10))
    
    # 1. Allele frequencies over time
    gens = list(range(len(freq_A)))
    
    ax1.plot(gens, freq_A, 'b-', linewidth=3, label='Frequency of A', marker='o', markersize=4)
    ax1.plot(gens, freq_a, 'r-', linewidth=3, label='Frequency of a', marker='s', markersize=4)
    ax1.axhline(0.5, color='gray', linestyle='--', alpha=0.5, label='Starting frequency')
    
    ax1.set_xlabel('Generation', fontsize=13, fontweight='bold')
    ax1.set_ylabel('Allele Frequency', fontsize=13, fontweight='bold')
    ax1.set_title(f'Allele Frequencies Over {generations} Generations\nSelection against aa: {selection_strength}%',
                 fontsize=15, fontweight='bold')
    ax1.legend(fontsize=12)
    ax1.grid(True, alpha=0.3)
    ax1.set_ylim(-0.05, 1.05)
    
    # 2. Genotype frequencies at final generation
    final_freq_A = freq_A[-1]
    final_freq_a = freq_a[-1]
    
    final_AA = final_freq_A ** 2
    final_Aa = 2 * final_freq_A * final_freq_a
    final_aa = final_freq_a ** 2
    
    genotypes = ['AA', 'Aa', 'aa']
    frequencies = [final_AA * 100, final_Aa * 100, final_aa * 100]
    colors = ['#FF6B6B', '#FFD93D', '#6BCF7F']
    
    bars = ax2.bar(genotypes, frequencies, color=colors, alpha=0.7, edgecolor='black', linewidth=2)
    ax2.set_ylabel('Frequency (%)', fontsize=13, fontweight='bold')
    ax2.set_title(f'Genotype Frequencies at Generation {generations}', fontsize=14, fontweight='bold')
    ax2.set_ylim(0, 100)
    ax2.grid(True, alpha=0.3, axis='y')
    
    for bar, freq in zip(bars, frequencies):
        height = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2., height + 2,
                f'{freq:.1f}%', ha='center', fontsize=11, fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    # Print interpretation
    print("\n" + "="*70)
    print("WHAT YOU'RE SEEING:")
    print("="*70)
    
    if selection_strength == 0:
        print("\nNO SELECTION: Hardy-Weinberg Equilibrium")
        print("  • Allele frequencies stay CONSTANT")
        print("  • Every generation: same Mendelian ratios")
        print("  • This is the baseline - evolution needs forces to act!")
        print("\n💡 KEY: Without selection/drift/migration, populations don't change!")
    else:
        print(f"\nWITH SELECTION: aa genotype has {selection_strength}% reduced survival")
        print(f"  • 'a' allele frequency changed from 0.500 to {final_freq_a:.3f}")
        print(f"  • 'a' is being selected AGAINST")
        print(f"  • But it's NOT eliminated (why? heterozygotes carry it!)")
        print("\n💡 KEY: Selection changes allele frequencies - that's evolution!")
        print("         But each generation still follows Mendelian inheritance!")
    
    print("\n" + "="*70)
    print("THE BRIDGE:")
    print("  Individual level: Each mating follows Mendelian ratios")
    print("  Population level: Allele frequencies can change due to selection")
    print("  Both are happening simultaneously!")
    print("="*70)

# Interactive widget
print("\n🧬 MULTI-GENERATION POPULATION SIMULATOR")
print("Watch how Mendelian inheritance + Selection = Evolution\n")

interact(visualize_generations,
         generations=IntSlider(min=1, max=100, step=1, value=20,
                             description='Generations:',
                             style={'description_width': 'initial'}),
         selection_strength=FloatSlider(min=0, max=50, step=5, value=0,
                                       description='Selection % against aa:',
                                       style={'description_width': 'initial'}));

---
## Summary: You've Made the Bridge!

### What You've Seen:

1. **ONE Family** → Mendelian genetics (deterministic probabilities)
2. **MANY Families** → Statistical patterns emerge (law of large numbers)
3. **POPULATION** → Hardy-Weinberg (frequencies instead of probabilities)
4. **OVER TIME** → Evolution (allele frequencies can change)

### The Key Insight:

**It's ALL the same mechanism!**
- At every scale: Mendelian inheritance is operating
- Family level: Focus on individual outcomes
- Population level: Focus on statistical patterns
- Over time: Add forces (selection, drift, migration)

### The Thinking Shift:

**Family Genetics:**
- "What will THIS child inherit?"
- Concrete, deterministic
- Small numbers
- Punnett squares

**Population Genetics:**
- "What patterns emerge across thousands?"
- Abstract, statistical  
- Large numbers
- Hardy-Weinberg

**But the DNA mechanism is IDENTICAL at both levels!**

---

### Practice Thinking at Both Levels:

**Scenario:** Cystic Fibrosis (CF)

**Family Level Question:**
"Two carriers have a child. What's the probability the child has CF?"
- Answer: 25% (Cc × Cc → ¼ cc)
- Tool: Punnett square

**Population Level Question:**
"Why is CF frequency 1 in 2,500 in Europeans?"
- Answer: q² = 1/2500, so q = 0.02, meaning 4% are carriers
- Tool: Hardy-Weinberg

**The Connection:**
The population frequency (1/2500) comes from millions of family Punnett squares over many generations!

---

🎉 **Congratulations! You can now think at both individual and population levels!**